In [0]:
source_table = 'pollution_per_country_pipeline_project.silver.countries_x_pollution'
source_table = spark.read.table(source_table)

co2_tonnes_per_capita = source_table.withColumn(
    'avg_co2_per_capita',
    (source_table['co2'] / source_table['population']) * 1000000
)

co2_tonnes_per_capita_filtered = co2_tonnes_per_capita.select("avg_co2_per_capita", "country", "year", "population")

co2_tonnes_per_capita_filtered.writeTo('pollution_per_country_pipeline_project.gold.co2_tonnes_per_capita_by_country').createOrReplace()



In [0]:
from pyspark.sql import functions as F

source_table = 'pollution_per_country_pipeline_project.silver.countries_x_pollution'
source_table = spark.read.table(source_table)

co2_tonnes_per_region = (
    source_table
    .groupBy("region","year")
    .agg(
        F.sum("co2").alias("co2_total_mt"),
        F.sum("population").alias("population_total")
    )
)

co2_tonnes_per_region_grouped = co2_tonnes_per_region.withColumn(
    'avg_co2_per_capita',
    (co2_tonnes_per_region['co2_total_mt'] / co2_tonnes_per_region['population_total']) * 1000000
)

co2_tonnes_per_region_grouped_filtered = co2_tonnes_per_region_grouped.select("region", "year", "avg_co2_per_capita")

co2_tonnes_per_region_grouped_filtered.writeTo('pollution_per_country_pipeline_project.gold.co2_tonnes_per_capita_by_region').createOrReplace()

'\n = source_table.withColumn(\n    \'avg_co2_per_capita\',\n    (sum(source_table[\'co2\']) / source_table[\'population\']) * 1000000\n).groupBy("region")\n\n'

In [0]:
for t in ["by_country", "by_region"]:
    name = f"pollution_per_country_pipeline_project.gold.co2_tonnes_per_capita_{t}"
    df = spark.read.table(name)
    print(name, df.count())
    df.printSchema()
    df.show(5)

pollution_per_country_pipeline_project.gold.co2_tonnes_per_capita_by_country 22386
root
 |-- avg_co2_per_capita: double (nullable = true)
 |-- country: string (nullable = true)
 |-- year: long (nullable = true)
 |-- population: double (nullable = true)

+--------------------+-----------+----+-----------+
|  avg_co2_per_capita|    country|year| population|
+--------------------+-----------+----+-----------+
|1.025208650464542...|South Korea|1905|  9754112.0|
|1.199063591288383...|  Indonesia|1889|3.3359365E7|
|2.278731990468975...|   Ethiopia|1949|1.7553622E7|
|2.281512782146843...|South Korea|1945|1.7532227E7|
|2.288812291242437...|   Ethiopia|1948|1.7476313E7|
+--------------------+-----------+----+-----------+
only showing top 5 rows
pollution_per_country_pipeline_project.gold.co2_tonnes_per_capita_by_region 1836
root
 |-- region: string (nullable = true)
 |-- year: long (nullable = true)
 |-- avg_co2_per_capita: double (nullable = true)

+--------------------+----+------------------